In [34]:

import os
import uuid

from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore
from langgraph.config import get_store
from langchain_core.messages import HumanMessage

load_dotenv(dotenv_path=".env")

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY is missing. Add it to .env")

@tool
def add_task(task: str) -> str:
    """Твоя задача - додати нове завдання до списку завдань."""
    task_id = str(uuid.uuid4())
    store = get_store()
    store.put(("tasks",), task_id, {"task": task})
    return f"Завдання додано: {task}"

@tool
def list_tasks() -> str:
    """Виводить всі завдання у списку."""
    store = get_store()
    rows = store.search(("tasks",))
    tasks = [
        f"{item.value.get('id', item.key)}: {item.value['task']}"
        for item in rows
    ]    
    return "Немає завдань" if not tasks else "Ось всі ваші завдання: " + ", ".join(tasks) 

@tool
def delete_task(task: str) -> str:
    """Видаляє завдання зі списку за описом або id."""
    store = get_store()
    rows = store.search(("tasks",))

    for item in rows:
        saved_task = item.value.get("task", "")
        saved_id = item.value.get("id", item.key)

        if task == saved_id:
            store.delete(("tasks",), item.key)
            return f"Завдання \"{saved_task}\" було видалено."

        if task.lower() in saved_task.lower() or saved_task.lower() in task.lower():
            store.delete(("tasks",), item.key)
            return f"Завдання \"{saved_task}\" було видалено."

    return f"Завдання не знайдено: {task}"

agent = create_agent(
    model=ChatOpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
        model="openai/gpt-4o-mini"
    ),
    tools=[add_task, list_tasks, delete_task],
    store=InMemoryStore(),
    system_prompt="""
    Ти працюєш з задачником.
    Відповідай ТІЛЬКИ українською мовою.
    Не додавай зайвих фраз.
    Кожна відповідь має бути короткою і в точному стилі:
    - для додавання: 'Завдання "<задача>" було успішно додано.'
    - для списку: 'Ось всі ваші завдання:\n1. <задача> [<id>]\n2. <задача> [<id>]'
    - для видалення: 'Завдання "<задача>" було видалено.'
    - для питання "Що залишилось?": 'У вас залишилася одна задача: <задача>.'
    """
)

tests = [
    "Додай: купити хліб",
    "Додай: подзвонити лікарю",
    "Покажи всі завдання",
    "Видали завдання про хліб",
    "Що залишилось?"
]

for i, test in enumerate(tests, 1):
    print(f"\n{'='*50}")
    print(f"TEST {i}: {test}")
    print("=" * 50)

    response = agent.invoke({
        "messages": [HumanMessage(content=test)]
    })
    last_message = response["messages"][-1].content
    print(f"ВІДПОВІДЬ: `{last_message}`")



TEST 1: Додай: купити хліб
ВІДПОВІДЬ: `Завдання "купити хліб" було успішно додано.`

TEST 2: Додай: подзвонити лікарю
ВІДПОВІДЬ: `Завдання "подзвонити лікарю" було успішно додано.`

TEST 3: Покажи всі завдання
ВІДПОВІДЬ: `Ось всі ваші завдання:
1. купити хліб [0d6c71e2-494c-444f-bbbe-ddef2711905a]
2. подзвонити лікарю [10a1d815-8943-4eb2-9a9a-c31c376746b8]`

TEST 4: Видали завдання про хліб
ВІДПОВІДЬ: `Завдання "купити хліб" було видалено.`

TEST 5: Що залишилось?
ВІДПОВІДЬ: `У вас залишилася одна задача: подзвонити лікарю.`
